# 7 — Expansion at European scale

Companion to **section 8**. Every zone is now extendable, one CO2 budget
covers the whole twelve-zone system, and the grid itself is a decision. The
note climbs a *ladder*: the basic system, then flexible vehicle charging,
then an industrial hydrogen sector with its electrolytic route.

Runtime: two solves of two to three minutes each at 168
segments. The full ladder at 1,095 segments is `pipeline/run_expansion.py`.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
# The investment models read their cost tables when imported, so a missing
# table shows up here rather than at the first solve.
try:
    from model import greenfield
except FileNotFoundError as missing:
    raise SystemExit(
        f"Missing {Path(missing.filename).name}. This file is not shipped with the "
        "repository: it is our reshaped subset of an external technology-cost "
        "database whose compiled outputs carry no single stated licence, so you "
        "build it yourself, once, with\n\n    python data/prepare.py --costs-only\n\n"
        "from the note's directory (one small download per vintage, at a pinned "
        "version). Notebooks 00-05 and 09 run without it."
    ) from None

costs = pd.read_csv(PROCESSED / f"technology_costs_full_{greenfield.FORWARD_HORIZON}.csv",
                    index_col="technology")
print(f"cost table: {len(costs)} technologies, {greenfield.FORWARD_HORIZON} vintage")

## The basic rung, unconstrained

Same construction as section 7 — the same candidate menu, the same weather
year — but in every zone at once, plus TYNDP's transmission candidates. Nothing
caps emissions yet; this solve gives the baseline the budget is a fraction of.

In [ ]:
from model import expansion
from run_greenfield import year_weather, REFERENCE_WEATHER_YEAR

HOURS = 168
NC = PROCESSED / "network_eur_bz_2024.nc"
weather = year_weather(REFERENCE_WEATHER_YEAR)

base = expansion.build(NC, costs, hours=HOURS, weather=weather, transmission=True)
expansion.solve(base)
unconstrained = expansion.system_emissions(base)
print(f"system emissions, unconstrained: {unconstrained/1e6:.1f} Mt/yr")
expansion.capacity(base).sum().div(1e3).round(1).sort_values(ascending=False).to_frame("built, all zones (GW)")

## A system-wide budget

Cut emissions to 25% of that. The dual is now the carbon price the *whole
system* faces — one price, every zone — which is what the ETS is. Look at
where the capacity goes: the model builds where the resource is, not where
the demand is, and buys wires to connect the two.

In [ ]:
n = expansion.build(NC, costs, hours=HOURS, weather=weather, transmission=True,
                    co2_budget=0.25 * unconstrained)
sigma = expansion.solve(n)
print(f"carbon price at 25% of unconstrained emissions: {sigma:.0f} EUR/t")

built = expansion.capacity(n).div(1e3).round(1)
built.loc[:, built.sum() > 0].sort_index()

## Your turn

1. Tighten the budget to 1% — the note's reference point — and watch what
   the last few per cent cost. (One more solve.)
2. The second rung: `ev=True` lets vehicle charging move within the day.
   Same budget, same demand — does the carbon price go up or down, and why
   does the note insist on comparing the two rungs against the *same*
   baseline?
3. The third rung: `hydrogen=True, p2x=True`. Then
   `expansion.hydrogen_balance(n)` says where the hydrogen came from —
   reformed from gas or electrolysed — and how hard the electrolysers ran.
   Section 8.4 of the note is about the number that decides whether the
   chain pays.
4. Compare `n.links` before and after: which transmission candidates were
   bought, and which zone pair's price spread did they close?

In [ ]:
# Try it here.